# Source verification and data audit

Research question: retrospective ranking and association of booking non-completion (cancellation or no-show); exact booking-creation-time prediction is unsupported because source fields are later snapshots. We preserve raw tokens, verify published schema/counts, reconstruct dates, and store eligible development sets separately from the final holdout. See `docs/COURSE_REQUIREMENTS.md` for the syllabus mapping and missing assignment screenshot. Exact equality does not prove duplicate error; no guest identifier is available.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/'src').is_dir() else Path.cwd().parent
assert (ROOT/'src/utils.py').exists(), 'Run from project root or notebooks/'
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from src import analysis
pd.set_option('display.max_colwidth', 100)


## Run this stage

The shared implementation in `src/analysis.py` produces the tables and figures. Run notebooks in numerical order on a fresh project. The source is part of the submitted, editable analysis.

In [2]:
result = analysis.audit()
display(result)

,partition,n,booking_min,booking_max
0,final_holdout,26565,2017-01-01,2017-08-31
1,left_boundary,8778,2013-06-24,2015-06-30
2,train,39865,2015-07-01,2016-06-29
3,train_unmatured,17610,2015-07-02,2016-06-30
4,validation,14088,2016-07-01,2016-12-30
5,validation_unmatured,12484,2016-07-01,2016-12-31


In [3]:
display(pd.read_csv(ROOT/'tables/raw_missing_tokens.csv').query('literal_NULL > 0 or literal_NA > 0'))

,column,dtype,literal_NULL,literal_NA,empty,n_unique
10,children,str,0,4,0,6
13,country,str,488,0,0,178
23,agent,str,16340,0,0,334
24,company,str,112593,0,0,353


In [4]:
display(pd.read_csv(ROOT/'tables/development_audit.csv'))

,check,count
0,eligible_development_rows,53953
1,zero_total_guests,91
2,zero_nights,520
3,negative_adr,0
4,adr_above_1000,1
5,adults_above_10,0
6,status_before_booking,0
7,negative_lead_time,0
8,train_exact_redundant_rows,11395
9,validation_exact_redundant_rows,1635


In [5]:
display(pd.read_csv(ROOT/'tables/prediction_time_leakage_audit.csv'))

,column,risk_category,primary_use,reason,source,enforced_exclusion
0,hotel,stable identity,conditional predictor,"Source hotel identity is stable; only two specific hotels, not replicated hotel types.","Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",False
1,is_canceled,outcome or outcome metadata,target,"Target, includes no-show in the published encoding; never a predictor.","Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",True
2,lead_time,conditional snapshot predictor,conditional predictor,"Definition supports booking_date = arrival_date - lead_time, but original entry timestamp and co...","Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",False
3,arrival_date_year,parsimony or redundant representation; not proven leakage,excluded predictor,Used to reconstruct calendar split; omit secular year from primary predictors.,"Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",True
4,arrival_date_month,parsimony or redundant representation; not proven leakage,excluded predictor,Replaced with fixed sine/cosine encoding; planned arrival may have changed.,"Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",True
5,arrival_date_week_number,parsimony or redundant representation; not proven leakage,excluded predictor,Redundant date component; omitted.,"Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",True
6,arrival_date_day_of_month,parsimony or redundant representation; not proven leakage,excluded predictor,Used for split reconstruction; omitted from predictors.,"Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",True
7,stays_in_weekend_nights,conditional snapshot predictor,conditional predictor,"Plausibly available at booking in an operational system, but this dataset stores later snapshots...","Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",False
8,stays_in_week_nights,conditional snapshot predictor,conditional predictor,"Plausibly available at booking in an operational system, but this dataset stores later snapshots...","Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",False
9,adults,conditional snapshot predictor,conditional predictor,"Plausibly available at booking in an operational system, but this dataset stores later snapshots...","Antonio et al. (2019), Table 1 and Section 2; project exclusion judgment",False


In [6]:
display(pd.read_csv(ROOT/'tables/target_semantics_development.csv'))

,partition,reservation_status,is_canceled,n
0,train,Canceled,1,11671
1,train,Check-Out,0,27607
2,train,No-Show,1,587
3,validation,Canceled,1,3545
4,validation,Check-Out,0,10399
5,validation,No-Show,1,144


## Interpretation boundary

These outputs concern a selected retrospective cohort. Review the report and leakage audit before interpreting model quality. The final holdout remains unscored.